# Proposed validation — review before it runs

**What this measures:** `heldout_accuracy` measures, through the real `ffmperative.ffmp` pipeline (extract → `search_best_trajectory` → `ast.parse` → `interpretor.evaluate`) with the generator stubbed to a frozen per-prompt rollout table in the repo's own test style, how many of 8 fixed eval items execute exactly the target tool-call sequence — the feature arm opts into the default-off `num_candidates=2` knob so cross-rollout recombination can repair split-partial rollouts, while the baseline arm (main) keeps the pre-change single rollout. Magnitudes are derived from the scoring arithmetic (`score = num_valid/num_steps + 0.01*num_valid`): baseline 3/8 = 0.375 (only complete-rollout items solvable), BES 6/8 = 0.75 (recombined full chains at 1.02–1.03 strictly dominate every split original ≤ 1.02), so the 0.625 = 5/8 bar demands ≥ 2 net repairs while the guardrail holds all 3 complete items at 1.0.

**Target metric:** `heldout_accuracy`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

This notebook is the only copy of the test: `.remyx/validation.yaml` points here, and the run executes these cells top to bottom.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the test Remyx drafted, split into cells; it was written as a script, so `__file__` is set to `eval/eval_bes_heldout_accuracy.py` (no such file is committed). This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_bes_heldout_accuracy.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
"""Held-out accuracy of BES forward evolutionary search vs the single-rollout baseline.

Drives the REAL public pipeline `ffmperative.ffmp` -> `interpretor
.extract_function_calls` -> `trajectory_search.search_best_trajectory` ->
`ast.parse` -> `interpretor.evaluate`. Only the generator (run_local/run_remote)
is replaced by a frozen per-prompt rollout table, mirroring the stub style of
tests/test_trajectory_search.py (3-tool lambda registry, no network, no ffmp
binary download). Success is judged by the executed tool-call sequence recorded
by the registry — never by trajectory_search's proxy score.

Frozen eval set (8 items): 3 complete (rollout 0 is the full target chain),
3 split (each rollout holds only part of the chain; no single rollout executes
the target), 2 unrepairable (unknown-tool rollouts only). Expected magnitudes:
baseline (main, 1 rollout/item) solves only complete items -> 3/8 = 0.375 and
holds the guardrail complete_items_accuracy = 1.0; on the PR head K=2
recombination splices the split parts into a candidate scoring 1+0.01*s, which
strictly dominates every 1.01 partial -> 6/8 = 0.75. Cost: 2 rollouts/item.
"""
import argparse
import inspect
import json
import os
import sys

In [ ]:
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, REPO_ROOT)

import ffmperative  # the changed call site: ffmp() itself

# Baseline/feature symmetry: the head's ffmp accepts num_candidates; on main it
# does not, so we fall back to the pre-change single-rollout call.
SUPPORTS_KW = "num_candidates" in inspect.signature(ffmperative.ffmp).parameters

In [ ]:
GI, SV, CV = "get_video_info", "sample_video", "compress_video"
GI_C, SV_C = "get_video_info(input='in.mp4')", "sample_video(video='in.mp4')"
CV_C = "compress_video(video='out.mp4')"

# (group, [rollout0, rollout1], target executed chain). The baseline arm
# consumes rollout 0 only; the head (num_candidates=2) consumes both.
ITEMS = [
    ("complete", [GI_C + "\n" + SV_C, SV_C], [GI, SV]),
    ("complete", [SV_C + "\n" + CV_C, CV_C], [SV, CV]),
    ("complete", [GI_C + "\n" + SV_C + "\n" + CV_C, GI_C], [GI, SV, CV]),
    ("split", [GI_C, SV_C], [GI, SV]),
    ("split", [SV_C, CV_C], [SV, CV]),
    ("split", [GI_C, SV_C + "\n" + CV_C], [GI, SV, CV]),
    ("unrepairable", ["rotate_video(input='in.mp4')", "flip_video(input='in.mp4')"], [CV]),
    ("unrepairable", ["rotate_video(input='in.mp4')", "rotate_video(input='in.mp4')"], [GI, CV]),
]
GROUPS = ("complete", "split", "unrepairable")

In [ ]:
def _tool(name, log):
    def call(*args, **kwargs):
        log.append(name)
        return "ok"
    return call

def run_pass(items, num_candidates):
    """Run ffmp over the frozen set; an item is solved iff the recorded
    executed call sequence equals its target chain."""
    solved = {g: [] for g in GROUPS}
    generate_calls = 0
    for group, rollouts, target in items:
        log, counter = [], {"n": 0}

        def gen(prompt, _r=rollouts, _c=counter):  # frozen rollout table
            i = min(_c["n"], len(_r) - 1)
            _c["n"] += 1
            return _r[i]

        ffmperative.run_local = gen   # repo-test-style stubbing: no network,
        ffmperative.run_remote = gen  # no ffmp binary, deterministic tables
        tools = {n: _tool(n, log) for n in (GI, SV, CV)}
        if SUPPORTS_KW:  # feature arm: search engaged (K = num_candidates)
            ffmperative.ffmp("frozen eval item", remote=False, tools=tools,
                             num_candidates=num_candidates)
        else:            # baseline arm: pre-change single-rollout behaviour
            ffmperative.ffmp("frozen eval item", remote=False, tools=tools)
        generate_calls += counter["n"]
        solved[group].append(log == target)
    return solved, generate_calls

In [ ]:
def _acc(flags):
    return round(sum(flags) / len(flags), 4) if flags else 0.0

def _overall(solved):
    return _acc([f for g in GROUPS for f in solved[g]])

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant")
    ap.add_argument("--ref")
    ap.add_argument("--seed", type=int)
    ap.parse_args()  # accepted and ignored

    items = [ITEMS[0], ITEMS[3]] if os.environ.get("REMYX_SMOKE") == "1" else ITEMS
    # Target measurement: K=2 on the head (<= 16 stubbed rollouts), K=1 on main.
    solved, rollouts = run_pass(items, 2)
    if SUPPORTS_KW:
        default_solved, _ = run_pass(items, 1)  # default path must be unchanged
        default_heldout = _overall(default_solved)
    else:
        default_heldout = _overall(solved)  # baseline: the default path IS this run
    print(json.dumps({
        "heldout_accuracy": _overall(solved),
        "complete_items_accuracy": _acc(solved["complete"]),
        "split_items_accuracy": _acc(solved["split"]),
        "unrepairable_items_accuracy": _acc(solved["unrepairable"]),
        "default_path_heldout_accuracy": default_heldout,
        "rollouts_consumed": rollouts,
    }))

if __name__ == "__main__":
    sys.exit(main())

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: compare
  ask: "Does the BES bidirectional evolutionary search in this PR improve the self-improvement loop's held-out accuracy over the repo's current baseline on a small fixed eval set, within one short run?"
benchmarks:
  - name: bes-heldout-accuracy
    suite: "eval/eval_bes_heldout_accuracy.py"
    scorer: heldout_accuracy
    baseline: main
    metrics:
      # Threshold derivation: N=8 = 3 complete-rollout + 3 split-partial + 2 unrepairable
      # items. Baseline (main, 1 rollout/prompt) can only solve complete items -> 3/8 = 0.375.
      # With K=2 the recombined full chain scores 1+0.01*s (1.02 two-step, 1.03 three-step)
      # and strictly dominates every split original (<= 1.02), so BES is expected 6/8 = 0.75.
      # 0.625 = 5/8 demands >= 2 net repairs over baseline with all complete items held,
      # tolerating one failed repair.
      - name: heldout_accuracy
        direction: max
        threshold: 0.625
        role: target
        bar: goal
        reads_as: "share of 8 frozen items whose executed call sequence matches the target; head runs num_candidates=2, main one rollout"
      # Baseline clears 1.0 by construction (rollout 0 is the full chain on these items);
      # the floor catches search degrading an already-complete rollout.
      - name: complete_items_accuracy
        direction: max
        threshold: 1.0
        role: guardrail
        bar: floor
        reads_as: "share of the 3 complete-rollout items still solved; baseline holds 1.0, so any drop is a selection regression"
      # Threshold derivation: 0.6667 = 2/3 split items repaired. With the guardrail
      # holding 3/3 complete and unrepairable at 0, target 5/8 is exactly >= 2 of 3
      # split items solved, so this bar mirrors the target's one-failed-repair
      # tolerance while isolating the recombination mechanism (baseline is 0.0 here).
      - name: split_items_accuracy
        direction: max
        threshold: 0.6667
        role: diagnostic
        reads_as: "share of the 3 split-partial items solved; 0.0 on baseline by construction - isolates the recombination repair"
      # Threshold derivation: 0.0 is the only correct value - these items' target
      # chains are unreachable from unknown-tool rollouts, so any nonzero share
      # means an unverified junk chain executed. Both arms are 0.0 by construction.
      - name: unrepairable_items_accuracy
        direction: min
        threshold: 0.0
        role: diagnostic
        reads_as: "share of 2 unknown-tool items whose unreachable target chain ran; expected 0.0 on both arms - any hit executed junk"
      # Threshold derivation: 0.375 = 3/8, the single-rollout result (baseline arm's
      # own run; head's num_candidates=1 run consumes rollout 0 only). The default
      # path must reproduce it, so the bar sits exactly on that value; both arms clear.
      - name: default_path_heldout_accuracy
        direction: max
        threshold: 0.375
        role: diagnostic
        reads_as: "accuracy of the num_candidates=1 default path; on head it must reproduce the baseline arm's single-rollout result"
      - name: rollouts_consumed
        direction: min
        threshold: 16
        role: cost
        reads_as: "stubbed generator calls used by the target measurement; budget 2 per item x 8 items = 16"
    policy:
      guardrail_veto: true
compute:
  tier: cpu
loop:
  max_iterations: 8
  fix_code: true
held_constant:
  - "frozen per-prompt recorded rollout tables, identical on both arms (baseline consumes rollout 0, feature consumes 2)"
  - "same 3-tool stub registry {get_video_info, sample_video, compress_video} recording executed calls, mirroring tests/test_trajectory_search.py"
  - "same public pipeline ffmperative.ffmp -> extract_function_calls -> search_best_trajectory -> ast.parse -> interpretor.evaluate"
  - "same item order and same success criterion (recorded executed call sequence == target chain), run_local stubbed so no network and no ffmp binary download"
avoid:
  - "live run_remote starcoder inference (network-bound, non-deterministic, unbounded 429 recursion) - the generator is stubbed per the repo's own test style"
  - "judging success by trajectory_search's proxy score instead of the executed call-sequence match (the 0.01/step depth bias prefers longer chains regardless of semantic correctness)"
  - "wall-clock timing on shared CPU"
  - "unpinned revisions or live APIs for anything the frozen tables already determine"
provenance:
  heldout_accuracy: "user_guidance"
  complete_items_accuracy: "maintainer_comment:@salma-remyx"
  split_items_accuracy: "maintainer_comment:@salma-remyx"
  unrepairable_items_accuracy: "maintainer_comment:@salma-remyx"
  default_path_heldout_accuracy: "inferred"
  rollouts_consumed: "inferred"
  held_constant: "repo_runner:tests/test_trajectory_search.py"
  suite: "synthesized"
report:
  headline: "BES forward evolution repairs split-partial rollouts on the frozen eval set, lifting held-out accuracy above the single-rollout baseline while the num_candidates=1 default path is unchanged"
  findings:
    - "Held-out accuracy on the frozen 8-item set is {heldout_accuracy} (single-rollout baseline expected 3/8 = 0.375)"
    - "{split_items_accuracy:%} of split-partial items are solved by recombination (0.0 on the baseline arm by construction)"
    - "{complete_items_accuracy:%} of complete-rollout items stay solved (guardrail floor 1.0)"
    - "{unrepairable_items_accuracy:%} of unrepairable items executed their unreachable target (expected 0.0)"
    - "The num_candidates=1 default path scores {default_path_heldout_accuracy}, matching the baseline arm's single-rollout behaviour"
    - "The target pass consumed {rollouts_consumed} stubbed rollouts (budget 16)"
  establishes:
    - "Forward candidate evolution (dense per-step grounding score + cross-rollout recombination before evaluate) improves held-out accuracy over the single-rollout path on a fixed split-partial rollout distribution, resting on {heldout_accuracy} against the 0.375 baseline"
    - "The PR's default path is behaviour-preserving at num_candidates=1, resting on {default_path_heldout_accuracy}"
  does_not_establish:
    - "The paper's full bidirectional claim - backward goal decomposition is unimplemented, so a pass supports only the forward-evolution half"
    - "Semantic correctness - the verifier is syntactic (known tool + bound names), so grounded-but-wrong chains still pass"
    - "Live-model accuracy or comparability to the paper's three open problem-solving benchmarks - the generator is a frozen recorded rollout table"
  not_measured:
    - "Sample-complexity reduction and entropy-shell escape (theoretical claims)"
    - "Post-training gains (the change is inference-time only)"
    - "Wall-clock latency, token cost, and best-of-N-with-recombine-off ablation"
  caveat: "BES's bidirectional half and real reward verifier are unimplemented; this PR's syntactic grounding check only repairs grounding failures and is inert on semantically wrong-but-grounded chains."
  next: "Re-run with a live or larger recorded rollout distribution, a semantic success verifier, and an explicit recombine_enabled=False best-of-N ablation to separate recombination from extra sampling."
```